# Generative Networks

Recurrent Neural Networks (RNNs) aur unke gated cell variants jaise Long Short Term Memory cells (LSTMs) aur Gated Recurrent Units (GRUs) ne language modeling ke liye ek mechanism diya, yaani woh words ka order seekh sakte hain aur sequence mein agla word predict kar sakte hain. Isse hum RNNs ko **generative tasks** ke liye use kar sakte hain, jaise ordinary text generation, machine translation, aur even image captioning.

Pichhle unit mein humne jis RNN architecture par baat ki thi, usme har RNN unit agle hidden state ko output ke roop mein produce karti thi. Lekin hum har recurrent unit mein ek aur output add kar sakte hain, jisse hum ek **sequence** output kar paayenge (jo original sequence ki length ke barabar hogi). Saath hi, hum aisi RNN units bhi use kar sakte hain jo har step par input accept nahi karti, balki sirf ek initial state vector leti hain aur phir outputs ki ek sequence produce karti hain.

Is notebook mein hum simple generative models par focus karenge jo text generate karne mein madad karte hain. Simplicity ke liye, chalo ek **character-level network** banate hain, jo letter by letter text generate karta hai. Training ke dauran humein koi text corpus lena hoga aur use letter sequences mein todna hoga.

In [ ]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset,test_dataset,classes,vocab = load_dataset()

## Character vocabulary banana

Character-level generative network banane ke liye, humein text ko words ke bajaye individual letters mein todna hoga. Yeh ek alag tokenizer define karke kiya ja sakta hai:

In [ ]:
def char_tokenizer(words):
    return list(words) #[word for word in words]

counter = collections.Counter()
for (label, line) in train_dataset:
    counter.update(char_tokenizer(line))
vocab = torchtext.vocab.vocab(counter)

vocab_size = len(vocab)
print(f"Vocabulary size = {vocab_size}")
print(f"Encoding of 'a' is {vocab.get_stoi()['a']}")
print(f"Character with code 13 is {vocab.get_itos()[13]}")

Chalo dekhte hain ki hum apne dataset ka text kaise encode kar sakte hain:

In [ ]:
def enc(x):
    return torch.LongTensor(encode(x,voc=vocab,tokenizer=char_tokenizer))

enc(train_dataset[0][1])

## Generative RNN ko train karna

Hum RNN ko text generate karne ke liye is tarah train karenge. Har step par hum `nchars` length ke characters ki ek string lenge aur network se kahenge ki har input character ke liye agla output character generate kare:

![Ek image jo 'HELLO' word ka RNN generation example dikha rahi hai.](../../../../../lessons/5-NLP/17-GenerativeNetworks/images/rnn-generate.png)

Real scenario ke hisaab se, hum kuch special characters bhi include karna chahenge, jaise *end-of-sequence* `<eos>`. Hamare case mein, hum network ko endless text generation ke liye train karna chahte hain, isliye hum har sequence ka size `nchars` tokens ke barabar fix karenge. Nateeja yeh hoga ki har training example mein `nchars` inputs aur `nchars` outputs honge (jo input sequence ko ek symbol left shift karke milte hain). Minibatch mein aisi kai sequences hongi.

Hum minibatch is tarah generate karenge ki har news text, jiski length `l` hai, se saare possible input-output combinations banayenge (aise `l-nchars` combinations honge). Yeh ek minibatch banayenge, aur har training step par minibatch ka size alag hoga.

In [ ]:
nchars = 100

def get_batch(s,nchars=nchars):
    ins = torch.zeros(len(s)-nchars,nchars,dtype=torch.long,device=device)
    outs = torch.zeros(len(s)-nchars,nchars,dtype=torch.long,device=device)
    for i in range(len(s)-nchars):
        ins[i] = enc(s[i:i+nchars])
        outs[i] = enc(s[i+1:i+nchars+1])
    return ins,outs

get_batch(train_dataset[0][1])

Ab hum generator network define karte hain. Yeh kisi bhi recurrent cell par based ho sakta hai jis par humne pichhle unit mein baat ki thi (simple, LSTM ya GRU). Hamare example mein hum LSTM use karenge.

Kyunki network characters ko input ke roop mein leta hai aur vocabulary size kaafi chhota hai, humein embedding layer ki zaroorat nahi hai. One-hot-encoded input seedha LSTM cell mein ja sakta hai. Lekin kyunki hum characters ke numbers ko input ke roop mein pass karte hain, humein unhe LSTM mein pass karne se pehle one-hot-encode karna hoga. Yeh `forward` pass ke dauran `one_hot` function ko call karke kiya jaata hai. Output encoder ek linear layer hogi jo hidden state ko one-hot-encoded output mein badal degi.

In [ ]:
class LSTMGenerator(torch.nn.Module):
    def __init__(self, vocab_size, hidden_dim):
        super().__init__()
        self.rnn = torch.nn.LSTM(vocab_size,hidden_dim,batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, s=None):
        x = torch.nn.functional.one_hot(x,vocab_size).to(torch.float32)
        x,s = self.rnn(x,s)
        return self.fc(x),s

Training ke dauran hum generate kiye gaye text ka sample dekhna chahte hain. Iske liye hum `generate` function define karenge, jo initial string `start` se shuru karke, length `size` ki output string generate karega.

Yeh aise kaam karta hai. Sabse pehle hum poori initial string ko network se pass karenge, aur output state `s` aur agla predicted character `out` paayenge. Kyunki `out` one-hot encoded hota hai, hum `argmax` use karke vocabulary mein character `nc` ka index nikalenge, aur `itos` use karke actual character pata karenge aur use characters ki resulting list `chars` mein add kar denge. Is process ko `size` baar repeat kiya jaata hai taaki zaroori ginti ke characters generate ho sakein.

In [ ]:
def generate(net,size=100,start='today '):
        chars = list(start)
        out, s = net(enc(chars).view(1,-1).to(device))
        for i in range(size):
            nc = torch.argmax(out[0][-1])
            chars.append(vocab.get_itos()[nc])
            out, s = net(nc.view(1,-1),s)
        return ''.join(chars)

Ab chalo training shuru karte hain! Training loop lagbhag hamare saare pichhle examples jaisa hi hai, lekin accuracy ke bajaye hum har 1000 epochs par generate kiya gaya sample text print karte hain.

Loss calculate karne ke tareeke par khaas dhyan dena hoga. Humein loss ko one-hot-encoded output `out` aur expected text `text_out` (jo character indexes ki list hai) ke basis par calculate karna hai. Khushkismati se, `cross_entropy` function unnormalized network output ko pehle argument ke roop mein aur class number ko doosre argument ke roop mein expect karta hai, jo hamare paas pehle se hai. Yeh minibatch size par automatic averaging bhi karta hai.

Hum training ko `samples_to_train` samples tak limit karte hain, taaki bahut zyada time na lage. Hum aapko encourage karte hain ki aap experiment karein aur lambi training try karein, ho sake toh kai epochs ke liye (aisi situation mein aapko is code ke aas-paas ek aur loop banana hoga).

In [ ]:
net = LSTMGenerator(vocab_size,64).to(device)

samples_to_train = 10000
optimizer = torch.optim.Adam(net.parameters(),0.01)
loss_fn = torch.nn.CrossEntropyLoss()
net.train()
for i,x in enumerate(train_dataset):
    # x[0] is class label, x[1] is text
    if len(x[1])-nchars<10:
        continue
    samples_to_train-=1
    if not samples_to_train: break
    text_in, text_out = get_batch(x[1])
    optimizer.zero_grad()
    out,s = net(text_in)
    loss = torch.nn.functional.cross_entropy(out.view(-1,vocab_size),text_out.flatten()) #cross_entropy(out,labels)
    loss.backward()
    optimizer.step()
    if i%1000==0:
        print(f"Current loss = {loss.item()}")
        print(generate(net))

Yeh example pehle se hi kaafi achha text generate karta hai, lekin ise kai tareekon se aur behtar banaya ja sakta hai:

* **Better minibatch generation**. Hamne training ke liye data jis tarah taiyaar kiya, woh ek sample se ek minibatch banana tha. Yeh ideal nahi hai, kyunki minibatches alag-alag size ke hote hain, aur kuch toh ban bhi nahi paate, kyunki text `nchars` se chhota hota hai. Saath hi, chhote minibatches GPU ko kaafi load nahi karte. Zyada samajhdaari yeh hogi ki saare samples se text ka ek bada chunk liya jaaye, phir saare input-output pairs banaye jaayein, unhe shuffle kiya jaaye, aur barabar size ke minibatches banaye jaayein.

* **Multilayer LSTM**. 2 ya 3 layers ke LSTM cells try karna samajhdaari hogi. Jaisa humne pichhle unit mein bataya tha, LSTM ki har layer text se kuch khaas patterns extract karti hai, aur character-level generator ke case mein hum ummeed kar sakte hain ki neeche ki LSTM layer letters ke groups (syllables) extract karne ke liye responsible hogi, aur upar ki layer words aur word combinations ke liye. Ise LSTM constructor mein layers ki ginti ka parameter pass karke aasaani se implement kiya ja sakta hai.

* Aap **GRU units** ke saath bhi experiment kar sakte hain aur dekh sakte hain ki kaun se behtar perform karte hain, aur **alag-alag hidden layer sizes** ke saath bhi. Bahut badi hidden layer overfitting ka kaaran ban sakti hai (jaise network exact text yaad kar lega), aur chhota size achhe results nahi de sakta.

## Soft text generation aur temperature

`generate` ki pichhli definition mein hum hamesha us character ko agla character chunte the jiski probability sabse zyada hoti thi. Iska result yeh hota tha ki text aksar baar-baar wahi character sequences mein "cycle" karne lagta tha, jaise is example mein:
```
today of the second the company and a second the company ...
```

Lekin agar hum agle character ki probability distribution dekhein, toh ho sakta hai ki kuch sabse high probabilities ke beech ka difference bahut bada na ho, jaise ek character ki probability 0.2 ho, aur doosre ki 0.19. Jaise, jab hum '*play*' sequence mein agla character dhoondh rahe hon, toh agla character space ya **e** (jaise word *player* mein) dono ho sakte hain.

Isse yeh nateeja nikalta hai ki hamesha highest probability wala character chunna "fair" nahi hai, kyunki doosre highest ko chunne se bhi meaningful text ban sakta hai. Zyada samajhdaari yeh hogi ki network output se di gayi probability distribution se **sample** karke characters chune jaayein.

Yeh sampling `multinomial` function use karke ki ja sakti hai, jo so-called **multinomial distribution** implement karta hai. Ek function jo is **soft** text generation ko implement karta hai, neeche define kiya gaya hai:

In [ ]:
def generate_soft(net,size=100,start='today ',temperature=1.0):
        chars = list(start)
        out, s = net(enc(chars).view(1,-1).to(device))
        for i in range(size):
            #nc = torch.argmax(out[0][-1])
            out_dist = out[0][-1].div(temperature).exp()
            nc = torch.multinomial(out_dist,1)[0]
            chars.append(vocab.get_itos()[nc])
            out, s = net(nc.view(1,-1),s)
        return ''.join(chars)
    
for i in [0.3,0.8,1.0,1.3,1.8]:
    print(f"--- Temperature = {i}\n{generate_soft(net,size=300,start='Today ',temperature=i)}\n")

Humne ek aur parameter **temperature** introduce kiya hai, jo yeh batane ke liye use hota hai ki humein highest probability se kitni sakhti se chipke rehna chahiye. Agar temperature 1.0 hai, toh hum fair multinomial sampling karte hain, aur jab temperature infinity ki taraf jaata hai - saari probabilities barabar ho jaati hain, aur hum agla character randomly chunte hain. Neeche diye gaye example mein hum dekh sakte hain ki jab hum temperature ko bahut badha dete hain toh text meaningless ho jaata hai, aur jab yeh 0 ke kareeb hota hai toh yeh "cycled" hard-generated text jaisa dikhta hai.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->